In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from torch.amp import GradScaler, autocast
import os
import numpy as np
import matplotlib.pyplot as plt


# для colab
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
class TimeIntervalEmbedding(nn.Module):
    # разница во времени между книгами
    def __init__(self, hidden_dim, max_interval=365):
        super().__init__()
        self.emb = nn.Embedding(max_interval + 1, hidden_dim, padding_idx=0)

    def forward(self, intervals):
        intervals = intervals.clamp(0, self.emb.num_embeddings - 1).long()
        return self.emb(intervals)

In [13]:
class SASRecTiSASRec(nn.Module):
    # модель с временными интервалами и признаком повтора

    def __init__(self, cnt_item, max_seq_len=30, hidden_dim=64,
                 num_heads=2, num_layers=2, dropout=0.2,
                 cnt_authors=0, cnt_categories=0, max_interval=365):
        super().__init__()

        self.max_seq_len = max_seq_len
        self.hidden_dim = hidden_dim

        self.item_emb = nn.Embedding(cnt_item + 1, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_dim)
        self.time_interval_emb = TimeIntervalEmbedding(hidden_dim, max_interval)
        self.repeat_emb = nn.Embedding(2, hidden_dim)

        if cnt_authors > 0:
            self.author_emb = nn.Embedding(cnt_authors + 1, hidden_dim, padding_idx=0)
        if cnt_categories > 0:
            self.category_emb = nn.Embedding(cnt_categories + 1, hidden_dim, padding_idx=0)

        self.emb_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=4 * hidden_dim, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.output = nn.Linear(hidden_dim, cnt_item + 1)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, item_ids, intervals, repeats, author_ids=None, category_ids=None):
        batch_size, seq_len = item_ids.shape

        item_emb = self.item_emb(item_ids)
        pos = torch.arange(seq_len, device=item_ids.device).unsqueeze(0)
        pos_emb = self.pos_emb(pos)
        interval_emb = self.time_interval_emb(intervals)
        repeat_emb = self.repeat_emb(repeats)

        x = item_emb + pos_emb + interval_emb + repeat_emb

        if hasattr(self, 'author_emb') and author_ids is not None:
            x = x + self.author_emb(author_ids)
        if hasattr(self, 'category_emb') and category_ids is not None:
            x = x + self.category_emb(category_ids)

        x = self.emb_norm(x)
        x = self.dropout(x)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=item_ids.device), diagonal=1
        ).bool()

        x = self.transformer(x, mask=causal_mask, is_causal=False)
        x = self.layer_norm(x)

        logits = self.output(x)
        return logits

In [14]:
def negative_sampling_loss(scores, positive_items, cnt_item, num_negatives=100):
    batch_size = scores.shape[0]
    pos_scores = scores[torch.arange(batch_size), positive_items]
    neg_ids = torch.randint(1, cnt_item, (batch_size, num_negatives), device=scores.device)
    neg_scores = scores.gather(1, neg_ids)
    pos_term = -F.logsigmoid(pos_scores)
    neg_term = -F.logsigmoid(-neg_scores).sum(dim=1)
    return (pos_term + neg_term).mean()

In [15]:
def ndcg_at_k(rel, pred, k=10):
    ndcg = 0.0
    if rel in pred[:k]:
        pred_list = list(pred[:k])
        score = pred_list.index(rel) + 1
        ndcg = 1.0 / np.log2(score + 1)
        return ndcg
    return ndcg


def recall_at_k(rel, pred, k=10):
    recall = 1.0 if rel in pred[:k] else 0.0
    return recall

In [16]:
# для colab
# data = torch.load('/content/drive/MyDrive/preprocessed_data_exp.pt')

data = torch.load('preprocessed_data_exp.pt')
train_inputs = data['train_inputs']
train_targets = data['train_targets']
train_authors = data['train_authors']
train_categories = data['train_categories']
train_times = data['train_timestamps']
validate_inputs = data['validate_inputs']
validate_targets = data['validate_targets']
validate_authors = data['validate_authors']
validate_categories = data['validate_categories']
validate_times = data['validate_timestamps']
cnt_item = data['cnt_item']
cnt_author = data['cnt_author']
cnt_category = data['cnt_category']

print(f"Train: {len(train_inputs)} примеров")
print(f"Validate: {len(validate_inputs)} примеров")

Train: 554526 примеров
Validate: 120934 примеров


In [18]:
# находим интервалы и повторы
def prepare_intervals_and_repeats(inputs, times):
    intervals = torch.zeros_like(times)
    mask = (times[:, 1:] > 0) & (times[:, :-1] > 0)
    intervals[:, 1:] = torch.where(mask, times[:, 1:] - times[:, :-1], torch.zeros_like(times[:, 1:]))

    repeats = torch.zeros_like(inputs)
    for b in range(len(inputs)):
        seen = set()
        for p in range(len(inputs[b])):
            if inputs[b][p] > 0:
                if inputs[b][p].item() in seen:
                    repeats[b][p] = 1
                else:
                    seen.add(inputs[b][p].item())

    return intervals.long(), repeats.long()


train_intervals, train_repeats = prepare_intervals_and_repeats(train_inputs, train_times)
val_intervals, val_repeats = prepare_intervals_and_repeats(validate_inputs, validate_times)

In [19]:
batch_size = 32
train_dataset = TensorDataset(train_inputs, train_targets,
                               train_intervals, train_repeats,
                               train_authors, train_categories)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = SASRecTiSASRec(
    cnt_item=cnt_item, max_seq_len=30,
    hidden_dim=64, num_heads=2, num_layers=2, dropout=0.2,
    cnt_authors=cnt_author, cnt_categories=cnt_category
).to(device)
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
Параметров: 20,731,933


In [ ]:
# валидация
def check_metrics_validate(model, validate_inputs, validate_targets,
                           val_intervals, val_repeats,
                           validate_authors, validate_categories,
                           k=10, batch_size=16):
    model.eval()
    ndcg_scores, recall_scores = [], []

    with torch.no_grad():
        for i in tqdm(range(0, len(validate_inputs), batch_size), desc="Validate"):
            batch_input = validate_inputs[i:i+batch_size].to(device)
            batch_target = validate_targets[i:i+batch_size]
            batch_intervals = val_intervals[i:i+batch_size].to(device)
            batch_repeats = val_repeats[i:i+batch_size].to(device)
            batch_authors = validate_authors[i:i+batch_size].to(device)
            batch_categories = validate_categories[i:i+batch_size].to(device)

            logits = model(batch_input, batch_intervals, batch_repeats,
                          batch_authors, batch_categories)
            scores = logits[:, -1, :]
            _, pred = torch.topk(scores, k=k, dim=1)
            pred = pred.cpu().numpy()

            for j in range(len(batch_target)):
                target = batch_target[j].item()
                ndcg_scores.append(ndcg_at_k(target, pred[j], k))
                recall_scores.append(recall_at_k(target, pred[j], k))

    model.train()
    return np.mean(ndcg_scores), np.mean(recall_scores)